In [1]:
# rca_agent_graph.py
from __future__ import annotations

import os, json, time, ast, sqlite3, re, glob
from typing import List, TypedDict, Literal, Any, Dict, Optional, Callable
from dataclasses import dataclass
from dotenv import load_dotenv

import pandas as pd

# --- LLM & Tools (LangChain) ---
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# --- LangGraph ---
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

# --- Langfuse tracing ---
from langfuse import Langfuse, get_client

/Users/gokul/miniconda3/envs/rca/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
METRICS_DIR = "metrics"
SQLITE_URI = "synthetic_data/synthetic_data.db"

In [3]:
# Use Gemini 1.5-flash
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
)

MAX_TURNS = 8

In [4]:
# Initialize Langfuse
langfuse = Langfuse(
  secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
  public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
  host="https://us.cloud.langfuse.com"
)


In [5]:
# Lazy df cache
_DF_CACHE: Dict[str, pd.DataFrame] = {}

In [6]:
# =========================
# DATA ACCESS HELPERS
# =========================

In [7]:
import json, re

def extract_json(text: str):
    try:
        return json.loads(text)
    except Exception:
        pass
    try:
        match = re.search(r"\{[\s\S]*\}|\[[\s\S]*\]", text)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass
    return None


In [8]:
def _load_csv(name: str) -> pd.DataFrame:
    """Load metrics CSV by stem or filename."""
    if not name.endswith(".csv"):
        name = f"{name}.csv"
    path = os.path.join(METRICS_DIR, name)
    if path in _DF_CACHE:
        return _DF_CACHE[path]
    if not os.path.exists(path):
        raise FileNotFoundError(f"CSV not found: {path}")
    df = pd.read_csv(path)
    _DF_CACHE[path] = df
    return df

In [9]:
def _list_metrics_files() -> List[str]:
    return sorted([os.path.basename(p) for p in glob.glob(os.path.join(METRICS_DIR, "*.csv"))])


In [10]:
def _connect_sqlite() -> sqlite3.Connection:
    if not os.path.exists(SQLITE_URI):
        raise FileNotFoundError(f"SQLite db not found at {SQLITE_URI}")
    conn = sqlite3.connect(SQLITE_URI, check_same_thread=False)
    return conn

In [11]:
# =========================
# SAFE EXEC FOR PANDAS
# =========================
_ALLOWED_ATTRS = {
    "pd": pd,
    "df": None,  # bound per call
}

_ALLOWED_CALLS = re.compile(
    r"(groupby|agg|mean|sum|count|size|min|max|std|var|median|quantile|nlargest|nsmallest|sort_values|"
    r"merge|join|concat|assign|pivot|pivot_table|rename|reset_index|set_index|drop|fillna|ffill|bfill|"
    r"astype|query|loc|iloc|head|tail|rolling|resample)"
)

In [12]:
def safe_pandas_exec(df: pd.DataFrame, code: str) -> pd.DataFrame | Any:
    """
    Execute a small pandas expression safely.
    Only allow attribute access and the whitelisted methods above.
    Example code the LLM may generate:
        result = df.query("region == 'North'").groupby('week').agg({'revenue':'sum'}).reset_index()
    Must assign to a variable named `result`.
    """
    # basic guards
    if "__" in code or "import" in code or "open(" in code or "exec(" in code or "eval(" in code:
        raise ValueError("Disallowed keywords in code")

    # rough whitelist check of method names
    for m in re.findall(r"\.(\w+)\(", code):
        if not _ALLOWED_CALLS.fullmatch(m):
            raise ValueError(f"Use of method '{m}' is not allowed")

    # parse AST to prohibit function defs, classes, loops, etc.
    tree = ast.parse(code, mode="exec")
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom, ast.With, ast.Lambda, ast.AsyncFunctionDef, ast.For, ast.While, ast.Try)):
            raise ValueError("Control structures not allowed in sandbox")

    # prepare namespace
    local_ns = {"pd": pd, "df": df.copy()}
    exec(compile(tree, filename="<safe>", mode="exec"), {}, local_ns)
    if "result" not in local_ns:
        raise ValueError("Your code must assign the output to a variable named `result`")
    return local_ns["result"]


In [13]:
# =========================
# TOOLS
# =========================

In [14]:
@tool("list_metrics")
def list_metrics_tool() -> str:
    """List available CSV files in the metrics folder."""
    files = _list_metrics_files()
    return json.dumps(files)

In [15]:
@tool("read_metric")
def read_metric_tool(csv_name: str, head: int = 5) -> str:
    """Read a metrics CSV by name (without path). Returns JSON with columns and a small preview."""
    df = _load_csv(csv_name)
    return json.dumps({"columns": df.columns.tolist(), "preview": df.head(head).to_dict(orient="records")})


In [16]:
@tool("run_sql")
def run_sql_tool(sql: str) -> str:
    """
    Run a read-only SQL query against synthetic_data.db and return JSON rows.
    Disallows mutating statements and ATTACH/PRAGMA.
    """
    stmt = sql.strip().lower()
    if not stmt.startswith("select"):
        raise ValueError("Only SELECT statements are allowed")
    for bad in ["attach", "pragma", "insert", "update", "delete", "drop", "alter", "create"]:
        if bad in stmt:
            raise ValueError(f"Disallowed keyword: {bad}")
    with _connect_sqlite() as conn:
        cur = conn.execute(sql)
        cols = [c[0] for c in cur.description]
        rows = [dict(zip(cols, r)) for r in cur.fetchall()]
    return json.dumps({"columns": cols, "rows": rows})

In [17]:
@tool("run_pandas")
def run_pandas_tool(csv_name: str, code: str) -> str:
    """
    Execute a safe pandas expression on a metrics CSV.
    The code MUST assign final output to a variable named `result`.
    Returns JSON: if result is a DataFrame, preview + columns; else str().
    """
    df = _load_csv(csv_name)
    out = safe_pandas_exec(df, code)
    if isinstance(out, pd.DataFrame):
        return json.dumps({"type": "dataframe", "columns": out.columns.tolist(), "preview": out.head(10).to_dict(orient="records")})
    else:
        return json.dumps({"type": "value", "value": str(out)})

In [18]:
TOOLS = [list_metrics_tool, read_metric_tool, run_sql_tool, run_pandas_tool]


In [19]:
# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """You are a Root-Cause Analysis (RCA) agent for retail/FMCG analytics.
You ONLY have access to:
1) metrics/*.csv via tools: list_metrics, read_metric, run_pandas
2) synthetic_data/synthetic_data.db via tool: run_sql

Rules:
- If user asks for a pattern (e.g., revenue dip/spike), FIRST read from metrics to extract precise pattern (segments, time window, magnitude).
- Then generate a concise checklist (≈5) of plausible causes that are testable with AVAILABLE columns/tables only.
- For each hypothesis, choose either run_sql or run_pandas, run a minimal query, and interpret numerically.
- Prefer smallest adequate tests. Never fabricate data or columns.
- Keep iterating until a high-confidence root cause is found or max steps reached.
- Summaries to the user must cite exact figures/time windows derived from observations.

You must not access any files outside metrics/ or the SQLite database.
"""

In [20]:
PATTERN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "User query: {query}\n\nUsing list_metrics/read_metric, determine the exact observed PATTERN (what metric, where, when, by how much). Output JSON with keys: metric, scope, window, magnitude, baseline, comparison, notes. Keep it short.")
])

CHECKLIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Pattern JSON: {pattern}\n\nList five testable hypotheses for root cause. Use columns/tables you actually have. Return as JSON list of short hypothesis strings.")
])

TEST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Pattern: {pattern}\nHypothesis: {hypothesis}\nAvailable metrics files: {metrics_files}\n\nWrite ONE tool call plan as JSON: {{'tool': 'run_sql'|'run_pandas', 'args': {{...}}, 'explain': 'what this test checks'}}.\n"
              "For run_pandas, include csv_name and a Python code snippet assigning to `result`.\nFor run_sql, include a SELECT statement.\nKeep it minimal.")
])

CRITIC_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Pattern: {pattern}\nEvidence so far:\n{evidence}\n\nDo we have a high-confidence root cause? Reply JSON: {{'confident': true|false, 'reason': '...'}}.")
])

FINAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Pattern: {pattern}\nEvidence:\n{evidence}\n\nWrite a crisp final answer for the user with:\n- pattern recap\n- top 1-2 quantified root causes with numbers\n- 2-3 next actions.\nNo JSON.")
])

In [21]:
# =========================
# GRAPH STATE
# =========================
class RCAState(TypedDict):
    query: str
    pattern: Optional[Dict[str, Any]]
    checklist: List[str]
    evidence: List[Dict[str, Any]]
    turn: int
    done: bool
    final_answer: Optional[str]
    errors: List[str]
    trace_id: Optional[str]

In [22]:
def _with_span(name: str) -> Callable:
    def decorator(func: Callable) -> Callable:
        def wrapper(state: RCAState):
            try:
                # Create a span for the node with proper metadata
                with langfuse.start_as_current_span(
                    name=name,
                    metadata={
                        "node": name, 
                        "query": state.get("query"),
                        "turn": state.get("turn", 0)
                    }
                ) as span:
                    # Store trace_id in state (only once)
                    if not state.get("trace_id"):
                        state["trace_id"] = span.trace_id
                    
                    try:
                        result = func(state)
                        return result
                    except Exception as e:
                        print(f"Span error in {name}: {e}")
                        raise
            except Exception as e:
                print(f"Langfuse error: {e}")
                raise
            finally:
                langfuse.flush()  # Ensure trace is sent
        return wrapper
    return decorator

In [23]:
@_with_span("router")
def router(state: RCAState) -> RCAState:
    # For now, everything routes to pattern detection
    return state

@_with_span("pattern_finder")
def pattern_finder(state: RCAState) -> RCAState:
    msg = PATTERN_PROMPT | llm
    resp = msg.invoke({"query": state["query"]})
    # Try to parse JSON
    parsed = extract_json(resp.content)
    if parsed is None:
        parsed = {"notes": f"Could not parse JSON. Raw: {resp.content}"}
    state["pattern"] = parsed
    return state

@_with_span("checklist_generator")
def checklist_generator(state: RCAState) -> RCAState:
    msg = CHECKLIST_PROMPT | llm
    resp = msg.invoke({"pattern": json.dumps(state["pattern"], ensure_ascii=False)})
    checklist = extract_json(resp.content)
    if isinstance(checklist, list):
        state["checklist"] = checklist[:5]
    else:
        # graceful fallback: extract bullet-like lines
        lines = [l.strip("- ").strip() for l in str(resp.content).splitlines() if l.strip()]
        state["checklist"] = lines[:5] if lines else []
    return state

@_with_span("hypothesis_tester")
def hypothesis_tester(state: RCAState) -> RCAState:
    if state["turn"] >= len(state["checklist"]):
        return state  # nothing to test
    hypothesis = state["checklist"][state["turn"]]
    plan_msg = TEST_PROMPT | llm
    metrics_files = _list_metrics_files()
    plan_resp = plan_msg.invoke({
        "pattern": json.dumps(state["pattern"]),
        "hypothesis": hypothesis,
        "metrics_files": metrics_files
    })
    # Parse plan
    plan = extract_json(plan_resp.content)
    if not isinstance(plan, dict):
        state["errors"].append(f"Plan parse failed for hypothesis {hypothesis}: {plan_resp.content}")
        state["turn"] += 1
        return state

    observation = {}
    try:
        if plan.get("tool") == "run_sql":
            observation = json.loads(run_sql_tool(plan["args"]["sql"]))
        elif plan.get("tool") == "run_pandas":
            observation = json.loads(run_pandas_tool(plan["args"]["csv_name"], plan["args"]["code"]))
        else:
            raise ValueError(f"Unknown tool {plan.get('tool')}")
        verdict = _interpret_observation(hypothesis, observation)
        state["evidence"].append({
            "hypothesis": hypothesis,
            "plan": plan,
            "observation": observation,
            "verdict": verdict,
        })
    except Exception as e:
        state["errors"].append(f"Hypothesis test failed: {e}")
    finally:
        state["turn"] += 1

    return state


In [24]:
def _interpret_observation(hypothesis: str, obs: Dict[str, Any]) -> str:
    # Very light auto-interpretation. The LLM will do the heavy lifting in CRITIC/FINAL.
    if obs.get("type") == "value":
        return f"Observed value: {obs['value']}"
    rows = obs.get("rows") or obs.get("preview") or []
    return f"Rows: {len(rows)}; sample: {rows[:3]}"

@_with_span("critic")
def critic(state: RCAState) -> RCAState:
    msg = CRITIC_PROMPT | llm
    resp = msg.invoke({
        "pattern": json.dumps(state["pattern"]),
        "evidence": json.dumps(state["evidence"])
    })
    judge = extract_json(resp.content)
    if isinstance(judge, dict):
        confidence = bool(judge.get("confident", False))
        state["done"] = confidence
        # Add more detailed logging
        print(f"Critic assessment: {'CONFIDENT' if confidence else 'NOT CONFIDENT'}")
        reason = judge.get("reason") or judge.get("reasoning")
        if reason:
            print(f"Reasoning: {reason}")
    else:
        print("Critic failed to parse response; continuing")
        state["done"] = False
    return state

@_with_span("generate_additional_hypotheses")
def generate_additional_hypotheses(state: RCAState) -> RCAState:
    """Generate new hypotheses when initial checklist is exhausted but we're not confident"""
    if state["turn"] >= len(state["checklist"]) and not state["done"]:
        print("Generating additional hypotheses based on current evidence...")
        
        # Create a prompt to generate new hypotheses based on current evidence
        additional_prompt = ChatPromptTemplate.from_template("""
Based on the pattern analysis and evidence gathered so far, generate 3 additional hypotheses to test:

Pattern: {pattern}
Evidence so far: {evidence}

Generate 3 new specific, testable hypotheses that could explain the root cause.
Focus on areas not yet explored based on the current evidence.

Return as JSON array: ["hypothesis1", "hypothesis2", "hypothesis3"]
""")
        
        msg = additional_prompt | llm
        resp = msg.invoke({
            "pattern": json.dumps(state["pattern"]),
            "evidence": json.dumps(state["evidence"])
        })
        
        new_hypotheses = extract_json(resp.content)
        if isinstance(new_hypotheses, list) and new_hypotheses:
            state["checklist"].extend(new_hypotheses[:3])  # Add up to 3 new hypotheses
            print(f"Added {len(new_hypotheses[:3])} new hypotheses to checklist")
        else:
            # fallback: pick plausible lines
            lines = [l.strip("- ").strip() for l in str(resp.content).splitlines() if l.strip()]
            if lines:
                state["checklist"].extend(lines[:3])
                print(f"Added {len(lines[:3])} fallback hypotheses to checklist")
            else:
                print("Failed to parse additional hypotheses")
    
    return state

@_with_span("decide_next")
def decide_next(state: RCAState) -> RCAState:
    # This function just updates state - the conditional edge handles routing
    # Add some logging to show decision making
    if state["done"]:
        print(f"Decision: Done - confidence reached")
    elif state["turn"] >= MAX_TURNS:
        print(f"Decision: Max turns ({MAX_TURNS}) reached")
    elif state["turn"] >= len(state["checklist"]):
        print(f"Decision: All checklist items tested")
    else:
        print(f"Decision: Continue testing - turn {state['turn']}/{len(state['checklist'])}")
    
    return state

@_with_span("finalize")
def finalize(state: RCAState) -> RCAState:
    msg = FINAL_PROMPT | llm
    resp = msg.invoke({
        "pattern": json.dumps(state["pattern"]),
        "evidence": json.dumps(state["evidence"])
    })
    state["final_answer"] = resp.content
    return state

In [25]:
# =========================
# BUILD GRAPH
# =========================
def build_graph():
    g = StateGraph(RCAState)

    g.add_node("router", router)
    g.add_node("pattern_finder", pattern_finder)
    g.add_node("checklist_generator", checklist_generator)
    g.add_node("hypothesis_tester", hypothesis_tester)
    g.add_node("critic", critic)
    g.add_node("generate_additional_hypotheses", generate_additional_hypotheses)
    g.add_node("decide_next", decide_next)
    g.add_node("finalize", finalize)

    g.add_edge(START, "router")
    g.add_edge("router", "pattern_finder")
    g.add_edge("pattern_finder", "checklist_generator")
    g.add_edge("checklist_generator", "hypothesis_tester")
    g.add_edge("hypothesis_tester", "critic")
    g.add_edge("critic", "decide_next")
    
    # Conditional edge from decide_next - loop back to hypothesis_tester, generate more hypotheses, or go to finalize
    def decide_next_route(state):
        if state["done"] or state["turn"] >= MAX_TURNS:
            return "finalize"
        elif state["turn"] >= len(state["checklist"]):
            return "generate_additional_hypotheses"
        else:
            return "hypothesis_tester"
    
    g.add_conditional_edges(
        "decide_next",
        decide_next_route,
        {
            "hypothesis_tester": "hypothesis_tester",
            "generate_additional_hypotheses": "generate_additional_hypotheses",
            "finalize": "finalize"
        }
    )
    
    # After generating additional hypotheses, go back to hypothesis_tester
    g.add_edge("generate_additional_hypotheses", "hypothesis_tester")
    
    g.add_edge("finalize", END)

    return g.compile()


In [26]:
# =========================
# ENTRYPOINT
# =========================
def run_rca(query: str) -> Dict[str, Any]:
    # Create a main trace for the entire RCA process
    with langfuse.start_as_current_span(
        name="rca_analysis",
        metadata={"query": query, "type": "root_cause_analysis"}
    ) as main_span:
        app = build_graph()
        init: RCAState = {
            "query": query,
            "pattern": None,
            "checklist": [],
            "evidence": [],
            "turn": 0,
            "done": False,
            "final_answer": None,
            "errors": [],
            "trace_id": main_span.trace_id,  # Set trace_id from main span
        }
        
        print(f"Starting RCA analysis for: {query}")
        print(f"Trace ID: {main_span.trace_id}")
        
        out = app.invoke(init)
        
        # Log final results
        print({
            "final_answer": out["final_answer"],
            "total_turns": out["turn"],
            "evidence_count": len(out["evidence"]),
            "errors_count": len(out["errors"])
        })
        
        return {
            "answer": out["final_answer"],
            "pattern": out["pattern"],
            "evidence": out["evidence"],
            "errors": out["errors"],
            "turns": out["turn"],
            "trace_id": out.get("trace_id"),
        }

In [27]:
# Test the improved RCA system
if __name__ == "__main__":
    # Build the graph
    app = build_graph()
    
    # Test query
    test_query = "Sales dropped significantly in Q3 2011"
    
    print("=" * 60)
    print("TESTING IMPROVED RCA SYSTEM")
    print("=" * 60)
    print(f"Query: {test_query}")
    print()
    
    # Run the RCA analysis
    result = run_rca(test_query)
    
    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)
    print(f"Answer: {result['answer']}")
    print(f"Total turns: {result['turns']}")
    print(f"Evidence collected: {len(result['evidence'])}")
    print(f"Errors: {len(result['errors'])}")
    print(f"Trace ID: {result['trace_id']}")
    
    if result['errors']:
        print("\nErrors encountered:")
        for error in result['errors']:
            print(f"  - {error}")
    
    print("\nEvidence summary:")
    for i, evidence in enumerate(result['evidence'], 1):
        print(f"  {i}. {evidence['hypothesis']}")
        print(f"     Verdict: {evidence['verdict']}")


TESTING IMPROVED RCA SYSTEM
Query: Sales dropped significantly in Q3 2011

Starting RCA analysis for: Sales dropped significantly in Q3 2011
Trace ID: d287fc22fbef325a6f30877793ce9488


/var/folders/ms/4rph23p959v4btjthd5yl64w0000gn/T/ipykernel_76580/4066356611.py:54: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  observation = json.loads(run_pandas_tool(plan["args"]["csv_name"], plan["args"]["code"]))


Critic failed to parse response; continuing
Decision: Continue testing - turn 1/5
Critic failed to parse response; continuing
Decision: Continue testing - turn 2/5
Critic failed to parse response; continuing
Decision: Continue testing - turn 3/5
Critic failed to parse response; continuing
Decision: Continue testing - turn 4/5
Critic failed to parse response; continuing
Decision: All checklist items tested
Generating additional hypotheses based on current evidence...
Added 3 new hypotheses to checklist
Critic failed to parse response; continuing
Decision: Continue testing - turn 6/8
Critic failed to parse response; continuing
Decision: Continue testing - turn 7/8


GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT

In [ ]:
q = "Find the pattern in revenue from 2011 jan to march and identify the root cause."
result = run_rca(q)
print(result["answer"])